### 결측치 처리
결측치?  
데이터 수집 문제 또는 데이터 입력 오류와 같은 다양한 이유로 관측되어야 할 값을 얻지 못한 데이터

In [13]:
import pandas as pd
import numpy as np

In [14]:
# 데이터프레임 생성
data = {
    'age': [25, np.nan, 30, 35, np.nan, 40],
    'salary': [50000, 55000, np.nan, 60000, 65000, np.nan],
    'height': [160, np.nan, 170, 175, 180, np.nan]
}

df = pd.DataFrame(data)
df

,age,salary,height
0,25.0,50000.0,160.0
1,NaN,55000.0,NaN
2,30.0,NaN,170.0
3,35.0,60000.0,175.0
4,NaN,65000.0,180.0
5,40.0,NaN,NaN


In [15]:
# age 컬럼의 결측치는 선형 보간법(Linear Interpolation)을 사용하여 처리
df['age'] = df['age'].interpolate(method='linear')
# salary 컬럼의 결측치는 평균값으로 대체
df['salary'] = df['salary'].fillna(df['salary'].mean())
# height 컬럼의 결측치는 중앙값으로 대체
df['height'] = df['height'].fillna(df['height'].median())
df

,age,salary,height
0,25.0,50000.0,160.0
1,27.5,55000.0,172.5
2,30.0,57500.0,170.0
3,35.0,60000.0,175.0
4,37.5,65000.0,180.0
5,40.0,57500.0,172.5


In [16]:
# 데이터프레임 생성
data2 = {
    'age': [25, np.nan, 30, 35, np.nan, 40],
    'salary': [50000, np.nan, np.nan, 60000, 65000, np.nan],
    'height': [160, np.nan, 170, 175, 180, np.nan]
}

df2 = pd.DataFrame(data2)
df2

,age,salary,height
0,25.0,50000.0,160.0
1,NaN,NaN,NaN
2,30.0,NaN,170.0
3,35.0,60000.0,175.0
4,NaN,65000.0,180.0
5,40.0,NaN,NaN


In [17]:
# 모든 컬럼에 결측치가 포함된 행 삭제
df2.dropna(how='all')

,age,salary,height
0,25.0,50000.0,160.0
2,30.0,NaN,170.0
3,35.0,60000.0,175.0
4,NaN,65000.0,180.0
5,40.0,NaN,NaN


In [18]:
df2.dropna(axis=1)

""
0
1
2
3
4
5


In [19]:
df2

,age,salary,height
0,25.0,50000.0,160.0
1,NaN,NaN,NaN
2,30.0,NaN,170.0
3,35.0,60000.0,175.0
4,NaN,65000.0,180.0
5,40.0,NaN,NaN


In [20]:
df2.dropna(subset=['salary'])

,age,salary,height
0,25.0,50000.0,160.0
3,35.0,60000.0,175.0
4,NaN,65000.0,180.0


In [21]:
df2

,age,salary,height
0,25.0,50000.0,160.0
1,NaN,NaN,NaN
2,30.0,NaN,170.0
3,35.0,60000.0,175.0
4,NaN,65000.0,180.0
5,40.0,NaN,NaN


In [22]:
# inplace=True : 원본을 직접 수정한다.
df2.dropna(subset=['salary'], inplace=True)

In [23]:
df2

,age,salary,height
0,25.0,50000.0,160.0
3,35.0,60000.0,175.0
4,NaN,65000.0,180.0


### 이상치 처리
이상치(Outlier)?
- 정상 데이터 범주에서 크게 벗어난 값

In [24]:
# 분위수 계산 예시
data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
q1 = np.quantile(data, 0.5) # 50% 분위수 (중앙값)
q1

np.float64(5.5)

In [ ]:
# 데이터프레임 생성
df = pd.DataFrame({
    'A': [1, 2, 3, 999],
    'B': [4, 5, 6, 777]
})
df

,A,B
0,1,4
1,2,5
2,3,6
3,999,777


In [ ]:
# apply(): 데이터프레임에 특정 함수를 각 행 또는 열에 적용
df2 = df.apply(lambda x: x.sum(), axis=0) # 열 단위로 합계 계산
df2

A    1005
B     792
dtype: int64

In [27]:
df3 = df.apply(lambda x: x.sum(), axis=1) # 행 단위로 합계 계산
df3

0       5
1       7
2       9
3    1776
dtype: int64

In [29]:
# 평균값, 표준편차 계산 예시
data2 = [1, 2, 3, 4, 5]
mean1 = np.mean(data2) # 평균값
mean1

np.float64(3.0)

In [30]:
# standard deviation
std1 = np.std(data2) # 표준편차
std1

np.float64(1.4142135623730951)

In [32]:
df.columns

Index(['A', 'B'], dtype='str')

In [35]:
df['B'].quantile(0.25)

np.float64(4.5)

In [36]:
df['B'].quantile(0.75)

np.float64(5.5)

In [ ]:
# 이상치 판단 기준 설정하기
# 이상치 대체하기 - IQR 기반
# IQR 기반 이상치를 판단하여 이상치로 판단된 경우 NaN으로 대체하는 함수
def detect_outliers(df):
    # 각 컬럼에 대해 IQR 계산
    for col in df.columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # 이상치 처리: 이상치가 있으면 해당 값들을 NaN으로 대체
        # A: 0 ~ 4
        # B: 3 ~ 6
        df[col] = df[col].apply(lambda x: x if lower_bound <= x <= upper_bound else None)

    return df

# 이상치 처리
df_cleaned = detect_outliers(df)
df_cleaned

,A,B
0,1.0,4.0
1,2.0,5.0
2,3.0,6.0
3,NaN,NaN


In [40]:
# 데이터프레임 생성
df = pd.DataFrame({
    'A': [1, 2, 3, 999],
    'B': [4, 5, 6, 777]
})
df

,A,B
0,1,4
1,2,5
2,3,6
3,999,777


In [41]:
df['A'].mean()

np.float64(251.25)

In [42]:
df['A'].std()

np.float64(498.50066867223626)

In [ ]:
# 절대값: 음수를 양수로 반환
abs(-1)

1

In [ ]:
# 이상치 대체하기 - Z-Score 기반
# Z-Score 기반으로 임계치(1)을 벗어날 경우, NaN으로 대체하는 함수
def detect_outliers_zscore(df, threshold=1):
    for col in df.columns:
        mean = df[col].mean()
        std_dev = df[col].std()

        # Z-Score 계산
        # z_scores = (df[col] - mean) / std_dev

        # 이상치 처리: Z-Score의 절대값이 threashold보다 큰 값들을 NaN으로 대체
        df[col] = df[col].apply(lambda x: x if abs((x - mean) / std_dev) <= threshold else None)

    return df

# 이상치 처리
df_cleaned = detect_outliers_zscore(df)
df_cleaned

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().